In [1]:
def make_graph():
  graph = {}
  return graph

In [2]:
def add_node(graph, node):
  graph[node] = set()

In [3]:
def add_edge(graph, node1, node2):
  if node1 not in graph:
    add_node(graph, node1)
  if node2 not in graph:
    add_node(graph, node2)
  graph[node1].add(node2)
  graph[node2].add(node1)

In [4]:
def remove_edge(graph, node1, node2):
  if node1 in graph and node2 in graph:
    graph[node1].remove(node2)
    graph[node2].remove(node1)

In [5]:
def all_nodes(graph):
  return list(graph.keys())

In [6]:
def neighbors(graph, node):
  return graph[node]

In [7]:
def initialize_labels(graph):
  labels = {}
  for node in graph.keys():
    labels[node] = node
  return labels

In [8]:
def count_neighbor_labels(graph, node, labels):
  node_neighbors = neighbors(graph, node)
  label_counts = {}
  for neighbor in node_neighbors:
    neighbor_label = labels[neighbor]
    if neighbor_label in label_counts:
      label_counts[neighbor_label] += 1
    else:
      label_counts[neighbor_label] = 1

  return label_counts

In [9]:
import random

def frequent_neighbor_label(neighbor_labels):
  max_count = max(neighbor_labels.values())
  candidates = []
  for label, count in neighbor_labels.items():
    if count == max_count:
      candidates.append(label)
  return random.choice(candidates)

In [10]:
import random

def async_iteration(graph, labels):
  all_nodes_list = all_nodes(graph)
  random.shuffle(all_nodes_list)

  for node in all_nodes_list:
    labels_counts = count_neighbor_labels(graph, node, labels)
    new_frequent_label = frequent_neighbor_label(labels_counts)

    if new_frequent_label is not None:
      labels[node] = new_frequent_label

  return labels

In [11]:
def stopping_condition(graph, labels):
  all_nodes_list = all_nodes(graph)
  for node in all_nodes_list:
    label_counts = count_neighbor_labels(graph, node, labels)

    #if there are no neighbors for a node
    if not label_counts:
      continue

    max_count = max(label_counts.values())
    current_label = labels[node]
    current_label_count = label_counts.get(current_label, 0)

    if current_label_count < max_count:
      return False

  return True

In [12]:
def label_propagation_algorithm(graph, max_iterations=100):
  labels = initialize_labels(graph)
  converged = False

  for iteration in range(max_iterations):
    labels = async_iteration(graph, labels)

    if stopping_condition(graph, labels):
      converged = True
      return labels, iteration + 1, converged

  return labels, max_iterations, converged

In [13]:
def group_by_label(labels):
    communities = {}

    for node, label in labels.items():
        if label not in communities:
            communities[label] = []

        communities[label].append(node)

    return communities, len(communities.keys())

In [14]:
def calculate_fsame(labels_1, labels_2):
    nodes = list(labels_1.keys())
    n = len(nodes)

    overlap = {}

    for node in nodes:
        label_1 = labels_1[node]
        label_2 = labels_2[node]

        if label_1 not in overlap:
            overlap[label_1] = {}

        if label_2 not in overlap[label_1]:
            overlap[label_1][label_2] = 0

        overlap[label_1][label_2] += 1

    row_max_sum = 0
    for label_1 in overlap:
        row_max_sum += max(overlap[label_1].values())

    labels_2_unique = set(labels_2.values())

    column_max_sum = 0
    for label_2 in labels_2_unique:
        column_values = []

        for label_1 in overlap:
            column_values.append(overlap[label_1].get(label_2, 0))

        column_max_sum += max(column_values)

    fsame = 0.5 * (row_max_sum + column_max_sum) * 100 / n

    return fsame

In [15]:
def calculate_jaccard(labels_1, labels_2):
    nodes = list(labels_1.keys())

    a = 0
    b = 0
    c = 0

    for i in range(len(nodes)):
        for j in range(i + 1, len(nodes)):
            node_i = nodes[i]
            node_j = nodes[j]

            same_in_1 = labels_1[node_i] == labels_1[node_j]
            same_in_2 = labels_2[node_i] == labels_2[node_j]

            if same_in_1 and same_in_2:
                a += 1
            elif same_in_1 and not same_in_2:
                b += 1
            elif not same_in_1 and same_in_2:
                c += 1

    if a + b + c == 0:
        return 0

    return a / (a + b + c)

In [16]:
test_graph = make_graph()

# Community 1
add_edge(test_graph, 'A', 'B')
add_edge(test_graph, 'A', 'C')
add_edge(test_graph, 'A', 'D')
add_edge(test_graph, 'B', 'C')
add_edge(test_graph, 'B', 'D')
add_edge(test_graph, 'C', 'D')

# Community 2
add_edge(test_graph, 'E', 'F')
add_edge(test_graph, 'E', 'G')
add_edge(test_graph, 'E', 'H')
add_edge(test_graph, 'F', 'G')
add_edge(test_graph, 'F', 'H')
add_edge(test_graph, 'G', 'H')

# Bridge between communities
add_edge(test_graph, 'D', 'E')

In [17]:
final_labels, iterations, converged = label_propagation_algorithm(test_graph)

print("Final labels:", final_labels)
print("Iterations:", iterations)
print("Converged:", converged)

communities, num_communities = group_by_label(final_labels)
print("Communities:", communities, "Number of communities: ", num_communities)

Final labels: {'A': 'D', 'B': 'D', 'C': 'D', 'D': 'D', 'E': 'F', 'F': 'F', 'G': 'F', 'H': 'F'}
Iterations: 2
Converged: True
Communities: {'D': ['A', 'B', 'C', 'D'], 'F': ['E', 'F', 'G', 'H']} Number of communities:  2


In [18]:
karate_graph = make_graph()

karate_edges = [
    (1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (1, 7), (1, 8), (1, 9),
    (1, 11), (1, 12), (1, 13), (1, 14), (1, 18), (1, 20), (1, 22), (1, 32),

    (2, 3), (2, 4), (2, 8), (2, 14), (2, 18), (2, 20), (2, 22), (2, 31),

    (3, 4), (3, 8), (3, 9), (3, 10), (3, 14), (3, 28), (3, 29), (3, 33),

    (4, 8), (4, 13), (4, 14),

    (5, 7), (5, 11),

    (6, 7), (6, 11), (6, 17),

    (7, 17),

    (9, 31), (9, 33), (9, 34),

    (10, 34),

    (14, 34),

    (15, 33), (15, 34),

    (16, 33), (16, 34),

    (19, 33), (19, 34),

    (20, 34),

    (21, 33), (21, 34),

    (23, 33), (23, 34),

    (24, 26), (24, 28), (24, 30), (24, 33), (24, 34),

    (25, 26), (25, 28), (25, 32),

    (26, 32),

    (27, 30), (27, 34),

    (28, 34),

    (29, 32), (29, 34),

    (30, 33), (30, 34),

    (31, 33), (31, 34),

    (32, 33), (32, 34),

    (33, 34)
]

for node1, node2 in karate_edges:
    add_edge(karate_graph, node1, node2)

In [19]:
final_labels_1, iterations_1, converged_1 = label_propagation_algorithm(karate_graph)

print("Final labels:", final_labels_1)
print("Iterations:", iterations_1)
print("Converged:", converged_1)

communities_1, num_communities_1 = group_by_label(final_labels_1)
print("Communities:", communities_1)
print("Number of communities:", num_communities_1)

Final labels: {1: 1, 2: 1, 3: 34, 4: 1, 5: 1, 6: 1, 7: 1, 8: 1, 9: 34, 11: 1, 12: 1, 13: 1, 14: 1, 18: 1, 20: 1, 22: 1, 32: 34, 31: 34, 10: 34, 28: 34, 29: 34, 33: 34, 17: 1, 34: 34, 15: 34, 16: 34, 19: 34, 21: 34, 23: 34, 24: 34, 26: 34, 30: 34, 25: 34, 27: 34}
Iterations: 3
Converged: True
Communities: {1: [1, 2, 4, 5, 6, 7, 8, 11, 12, 13, 14, 18, 20, 22, 17], 34: [3, 9, 32, 31, 10, 28, 29, 33, 34, 15, 16, 19, 21, 23, 24, 26, 30, 25, 27]}
Number of communities: 2


In [20]:
final_labels_2, iterations_2, converged_2 = label_propagation_algorithm(karate_graph)

print("Final labels:", final_labels_2)
print("Iterations:", iterations_2)
print("Converged:", converged_2)

communities_2, num_communities_2 = group_by_label(final_labels_2)
print("Communities:", communities_2)
print("Number of communities:", num_communities_2)

Final labels: {1: 8, 2: 8, 3: 8, 4: 8, 5: 8, 6: 8, 7: 8, 8: 8, 9: 34, 11: 8, 12: 8, 13: 8, 14: 8, 18: 8, 20: 8, 22: 8, 32: 34, 31: 34, 10: 34, 28: 34, 29: 34, 33: 34, 17: 8, 34: 34, 15: 34, 16: 34, 19: 34, 21: 34, 23: 34, 24: 34, 26: 34, 30: 34, 25: 34, 27: 34}
Iterations: 6
Converged: True
Communities: {8: [1, 2, 3, 4, 5, 6, 7, 8, 11, 12, 13, 14, 18, 20, 22, 17], 34: [9, 32, 31, 10, 28, 29, 33, 34, 15, 16, 19, 21, 23, 24, 26, 30, 25, 27]}
Number of communities: 2


In [21]:
fsame = calculate_fsame(final_labels_1, final_labels_2)
jaccard = calculate_jaccard(final_labels_1, final_labels_2)

print("fsame:", fsame)
print("Jaccard:", jaccard)

fsame: 97.05882352941177
Jaccard: 0.8865979381443299
